<a href="https://colab.research.google.com/github/PoojaYadav2344/reliability-estimation-qa/blob/main/Reliability_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reliability and Uncertainty Estimation Framework for Question Answering Transformer-based Systems

This notebook contains the implementation and experimental framework used in the study for evaluating the reliability and uncertainty characteristics of transformer-based question answering (QA) systems under both answerable and unanswerable question settings.

The study utilizes two benchmark QA datasets representing complementary evaluation scenarios:

1. **SQuAD v2.0 Dataset**  

2. **QuAC Dataset**

Multiple transformer-based QA architectures are evaluated, including:
- BERT,
- RoBERTa,
- ALBERT,
- and DistilBERT.

The notebook includes implementations codes for:
- Stochastic perturbation induced in the models's architecture via Monte Carlo Dropout (MCD)-based uncertainty estimation.
- perturbation induced in the input provided to the model via paraphrasing.


# Description of the Present Implementation

This notebook implements the uncertainty and reliability evaluation pipeline for transformer-based question answering models including RoBERTa, DistilBERT, ALBERT, and BERT Base. The code performs dataset preprocessing, question paraphrasing, QA inference, cosine similarity computation, F1 score evaluation, and statistical analysis of prediction consistency under paraphrased input perturbations using the QuAC dataset under the answerable question setting.

In [ ]:
import random
import torch
import nltk
import json
import statistics
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
from sentence_transformers import SentenceTransformer, util

# Download necessary NLTK resources
nltk.download('punkt')

# Load SBERT model for similarity computation
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Load paraphrasing model
paraphrase_pipeline = pipeline("text2text-generation", model="eugenesiow/bart-paraphrase")

# Function to paraphrase a question using the BART-based model
def paraphrase_with_bart(question):
    paraphrased_output = paraphrase_pipeline(question, max_length=100, num_return_sequences=1)
    return paraphrased_output[0]['generated_text']

# Function to calculate F1 score between predicted and correct answers
def compute_f1(predicted_answer, correct_answers):
    predicted_tokens = set(predicted_answer.lower().split())
    max_f1 = 0
    for correct_answer in correct_answers:
        correct_tokens = set(correct_answer.lower().split())
        common_tokens = predicted_tokens.intersection(correct_tokens)
        if len(common_tokens) == 0:
            continue
        precision = len(common_tokens) / len(predicted_tokens) if len(predicted_tokens) > 0 else 0
        recall = len(common_tokens) / len(correct_tokens) if len(correct_tokens) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        max_f1 = max(max_f1, f1)
    return max_f1

# Step 1: Load the SQuAD 2.0 dataset
with open('QuAC_dev.json', 'r') as f:
    quac_data = json.load(f)

# Extract dataset samples (contexts, questions, answers)
data = []
for article in quac_data['data']:
    for paragraph in article['paragraphs']:
        context = paragraph['context']
        for qa in paragraph['qas']:
            # Exclude questions with "CANNOTANSWER"
            if 'answers' in qa and len(qa['answers']) > 0 and qa['answers'][0]['text'] != "CANNOTANSWER":
                question = qa['question']
                correct_answers = [answer['text'] for answer in qa['answers']]
                data.append((context, question, correct_answers))

# Use random questions with answers from the dataset
subset = data

# Detect if CUDA is available
device = 0 if torch.cuda.is_available() else -1

# Function to load QA models
def load_qa_pipeline(model_path, device):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForQuestionAnswering.from_pretrained(model_path)
    return pipeline("question-answering", model=model, tokenizer=tokenizer, device=device)

# Load four QA models
models = {
    "RoBERTa": load_qa_pipeline("deepset/roberta-base-squad2", device),
    "DistilBERT": load_qa_pipeline("distilbert-base-uncased-distilled-squad", device),
    "ALBERT": load_qa_pipeline("twmkn9/albert-base-v2-squad2", device),
    "BERT Base": load_qa_pipeline("deepset/bert-base-cased-squad2", device),
}

# Metrics storage for all models
all_results = {
    model_name: {
        "answer_cosine_similarities": [],
        "f1_score_drops": [],
        "question_cosine_similarities_filtered": [],
        "f1_paraphrased_scores": []  # NEW
    }
    for model_name in models
}

# Process dataset with paraphrasing
for idx, (context, question, correct_answers) in enumerate(subset):
    paraphrased_question = paraphrase_with_bart(question)

    # Compute cosine similarity between original and paraphrased question
    original_question_embedding = sbert_model.encode(question, convert_to_tensor=True)
    paraphrased_question_embedding = sbert_model.encode(paraphrased_question, convert_to_tensor=True)
    question_similarity = util.cos_sim(original_question_embedding, paraphrased_question_embedding).item()

    # Continue processing only if similarity is between 0.75 and 0.98
    if 0.75 <= question_similarity <= 0.98:
        print(f"\nOriginal Question {idx + 1}: {question}")
        print(f"Paraphrased Question: {paraphrased_question}")
        print(f"Cosine Similarity (Original vs. Paraphrased Question): {question_similarity:.4f}")
        print(f"Correct Answer(s) from Dataset: {correct_answers}")

        for model_name, qa_pipeline in models.items():
            # Predict answer for original question
            original_prediction = qa_pipeline(question=question, context=context)
            original_predicted_answer = original_prediction["answer"]

            # Predict answer for paraphrased question
            paraphrased_prediction = qa_pipeline(question=paraphrased_question, context=context)
            paraphrased_predicted_answer = paraphrased_prediction["answer"]

            print(f"\n[{model_name}] Predicted Answer (Original): {original_predicted_answer}")
            print(f"[{model_name}] Predicted Answer (Paraphrased): {paraphrased_predicted_answer}")

            # Cosine similarity between paraphrased predicted answer and correct answers
            paraphrased_predicted_embedding = sbert_model.encode(paraphrased_predicted_answer, convert_to_tensor=True)
            for correct_answer in correct_answers:
                correct_answer_embedding = sbert_model.encode(correct_answer, convert_to_tensor=True)
                answer_similarity = util.cos_sim(paraphrased_predicted_embedding, correct_answer_embedding).item()
                all_results[model_name]["answer_cosine_similarities"].append(answer_similarity)

            # F1 scores
            f1_original = compute_f1(original_predicted_answer, correct_answers)
            f1_paraphrased = compute_f1(paraphrased_predicted_answer, correct_answers)
            f1_drop = f1_original - f1_paraphrased

            all_results[model_name]["f1_score_drops"].append(f1_drop)
            all_results[model_name]["f1_paraphrased_scores"].append(f1_paraphrased)  # NEW

            print(f"[{model_name}] Cosine Similarity (Paraphrased Predicted vs. Correct Answer): {answer_similarity:.4f}")
            print(f"[{model_name}] F1 Score (Original): {f1_original:.4f}")
            print(f"[{model_name}] F1 Score (Paraphrased): {f1_paraphrased:.4f}")
            print(f"[{model_name}] F1 Score Drop: {f1_drop:.4f}")

        # Store question similarity
        for model_name in models:
            all_results[model_name]["question_cosine_similarities_filtered"].append(question_similarity)

# Summary Results for All Models
print("\n===== Summary Results After Paraphrasing =====")
for model_name in models:
    cosine_sim_list = all_results[model_name]["answer_cosine_similarities"]
    f1_paraphrased_list = all_results[model_name]["f1_paraphrased_scores"]

    print(f"\n== {model_name} ==")
    print(f"Total Questions Processed: {len(cosine_sim_list)}")
    print(f"Cosine Similarities (Paraphrased Predicted Answers vs. Correct Answers): {cosine_sim_list}")
    print(f"F1 Score Drops: {all_results[model_name]['f1_score_drops']}")
    print(f"F1 Scores (Paraphrased vs. Ground Truth): {f1_paraphrased_list}")
    print(f"Question Cosine Similarities (Processed Questions): {all_results[model_name]['question_cosine_similarities_filtered']}")

    # Calculate and print means
    if cosine_sim_list:
        print(f"Mean Cosine Similarity (Answers): {statistics.mean(cosine_sim_list):.4f}")
    if f1_paraphrased_list:
        print(f"Mean F1 Score (Paraphrased vs. Ground Truth): {statistics.mean(f1_paraphrased_list):.4f}")

# Print the mean of F1 score drops across all models
f1_drop_all_models = []
for model_name in models:
    f1_drop_all_models.extend(all_results[model_name]['f1_score_drops'])

if f1_drop_all_models:
    print(f"\nMean F1 Score Drop (Across All Models): {statistics.mean(f1_drop_all_models):.4f}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Device set to use cuda:0


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at deepset/bert-base-cased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0



Original Question 2: did they have any children?
Paraphrased Question: Did they have children?
Cosine Similarity (Original vs. Paraphrased Question): 0.9638
Correct Answer(s) from Dataset: ['she gave birth to her daughter Sofia.', 'in 1975 and in November she gave birth to her daughter Sofia.', 'her daughter Sofia.', 'in November she gave birth to her daughter Sofia.']

[RoBERTa] Predicted Answer (Original): she gave birth to her daughter Sofia
[RoBERTa] Predicted Answer (Paraphrased): she gave birth to her daughter Sofia
[RoBERTa] Cosine Similarity (Paraphrased Predicted vs. Correct Answer): 0.9211
[RoBERTa] F1 Score (Original): 0.8571
[RoBERTa] F1 Score (Paraphrased): 0.8571
[RoBERTa] F1 Score Drop: 0.0000

[DistilBERT] Predicted Answer (Original): Sofia
[DistilBERT] Predicted Answer (Paraphrased): Sofia
[DistilBERT] Cosine Similarity (Paraphrased Predicted vs. Correct Answer): 0.5429
[DistilBERT] F1 Score (Original): 0.0000
[DistilBERT] F1 Score (Paraphrased): 0.0000
[DistilBERT] F

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Streaming output truncated to the last 5000 lines.
[ALBERT] F1 Score (Original): 0.5000
[ALBERT] F1 Score (Paraphrased): 0.5000
[ALBERT] F1 Score Drop: 0.0000

[BERT Base] Predicted Answer (Original): Chungking Express
[BERT Base] Predicted Answer (Paraphrased): Chungking Express
[BERT Base] Cosine Similarity (Paraphrased Predicted vs. Correct Answer): 0.9561
[BERT Base] F1 Score (Original): 1.0000
[BERT Base] F1 Score (Paraphrased): 1.0000
[BERT Base] F1 Score Drop: 0.0000

Original Question 5533: How did he get this reputation?
Paraphrased Question: How did he get his reputation?
Cosine Similarity (Original vs. Paraphrased Question): 0.9475
Correct Answer(s) from Dataset: ['His behavior during the filming of Mutiny on the Bounty (1962) seemed to bolster his reputation as a difficult star.', '" His behavior during the filming of Mutiny on the Bounty (1962) seemed to bolster his reputation as a difficult star.', 'during the filming of Mutiny on the Bounty (1962) seemed to bolster his r

# Description of the Present Implementation  

This notebook implements the uncertainty and reliability evaluation pipeline for transformer-based question answering model - RoBERTa. The code performs dataset preprocessing, QA inference, Monte Carlo Dropout sampling, cosine similarity computation, F1 score evaluation, and statistical analysis of prediction consistency across multiple stochastic forward passes using the SQuAD v2.0 dataset under the unanswerable question setting.

In [ ]:
import json
import random
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np

# Function to calculate F1 score between predicted and correct answers
def compute_f1(predicted_answer, correct_answer):
    if not predicted_answer and not correct_answer:  # Both predicted and correct are blank
        return 1.0
    predicted_tokens = set(predicted_answer.lower().split())
    correct_tokens = set(correct_answer.lower().split())
    common_tokens = predicted_tokens.intersection(correct_tokens)
    if len(common_tokens) == 0:
        return 0.0
    precision = len(common_tokens) / len(predicted_tokens) if len(predicted_tokens) > 0 else 0
    recall = len(common_tokens) / len(correct_tokens) if len(correct_tokens) > 0 else 0
    return 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# User-defined dropout rate
USER_DEFINED_DROPOUT_RATE = 0.10  # Adjust this value as needed

# Step 1: Load the SQuAD 2.0 dataset
with open('dev-v2.0.json', 'r') as f:
    squad_data = json.load(f)

# Extract dataset samples (contexts, questions, answers)
data = []
for article in squad_data['data']:
    for paragraph in article['paragraphs']:
        context = paragraph['context']
        for qa in paragraph['qas']:
            # Include only unanswerable questions (answers are an empty list [])
            if 'answers' in qa and len(qa['answers']) == 0:
                question = qa['question']
                correct_answer = ""  # Unanswerable questions have no correct answer
                data.append((context, question, correct_answer))

# Use unanswerable questions from the dataset
subset = data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_qa_pipeline(model_name, device):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)
    return pipeline("question-answering", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Load DistilBERT model instead of BERT Base
qa_pipeline = load_qa_pipeline("deepset/roberta-base-squad2", device)

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def enable_dropout(model, dropout_rate):
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = dropout_rate
            module.train()

# Enable dropout in the model
enable_dropout(qa_pipeline.model, USER_DEFINED_DROPOUT_RATE)

# Lists for storing final statistics
cosine_sim_before_dropout = []
cosine_sim_first_sample = []
avg_cosine_sim_list = []
var_cosine_sim_list = []
avg_f1_score_list = []
var_f1_score_list = []

for idx, (context, question, correct_answers) in enumerate(subset):
    print(f"\nQuestion {idx + 1}:")
    print(f"Context: {context}")
    print(f"Question: {question}")
    print(f"Correct Answers: {correct_answers}")

    print(f"\nModel: RoBERTa")

    # Predict answer using RoBERTa
    inputs = qa_pipeline.tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512).to(device)  # Move tensors to GPU
    with torch.no_grad():
      outputs = qa_pipeline.model(**inputs) # Use qa_pipeline.model instead of just model
      start_scores = outputs.start_logits
      end_scores = outputs.end_logits

    start_idx = torch.argmax(start_scores)
    end_idx = torch.argmax(end_scores) + 1
    predicted_answer = qa_pipeline.tokenizer.decode(inputs["input_ids"][0][start_idx:end_idx], skip_special_tokens=True) # Use qa_pipeline.tokenizer for decoding
    print(f"Predicted Answer: {predicted_answer}")

    # Step 4: Compute similarity with correct answer
    if not predicted_answer and not correct_answers:  # Both predicted and correct are blank
     max_similarity = 1.0
     f1 = 1.0
     correct_embedding = torch.zeros((1, sbert_model.get_sentence_embedding_dimension())).to(device)  # Placeholder tensor on the correct device
    else:
     predicted_embedding = sbert_model.encode(predicted_answer, convert_to_tensor=True, show_progress_bar=False).to(device)
     correct_embedding = sbert_model.encode(correct_answers, convert_to_tensor=True, show_progress_bar=False).to(device)
     max_similarity = util.cos_sim(correct_embedding, predicted_embedding).item()
     f1 = compute_f1(predicted_answer, correct_answers)

    print(f"Maximum Cosine Similarity Before MC Dropout: {max_similarity:.4f}")
    cosine_sim_before_dropout.append(float(max_similarity))

    num_samples = 10
    predicted_samples, cosine_similarities, f1_scores = [], [], []

    for i in range(num_samples):
       with torch.no_grad():
        outputs = qa_pipeline.model(**inputs)
        start_scores = outputs.start_logits
        end_scores = outputs.end_logits

        start_idx = torch.argmax(start_scores)
        end_idx = torch.argmax(end_scores) + 1
        answer_sample = qa_pipeline.tokenizer.decode(inputs["input_ids"][0][start_idx:end_idx], skip_special_tokens=True)
        predicted_samples.append(answer_sample)

        if not answer_sample and not correct_answers:
            similarity = 1.0
            f1 = 1.0
        else:
            sample_embedding = sbert_model.encode(answer_sample, convert_to_tensor=True)
            similarity = util.cos_sim(correct_embedding, sample_embedding).item()
            f1 = compute_f1(answer_sample, correct_answers)

        cosine_similarities.append(similarity)
        f1_scores.append(float(f1))

        if i == 0:
            cosine_sim_first_sample.append(float(similarity))

    print("\nPredicted Answers from MC Dropout Samples:")
    for i, sample in enumerate(predicted_samples, start=1):
        print(f"Sample {i}: {sample} (Cosine Similarity: {cosine_similarities[i-1]:.4f}, F1 Score: {f1_scores[i-1]:.4f})")

    avg_cosine_sim_list.append(float(np.mean(cosine_similarities)))
    var_cosine_sim_list.append(float(np.var(cosine_similarities)))
    avg_f1_score_list.append(float(np.mean(f1_scores)))
    var_f1_score_list.append(float(np.var(f1_scores)))

print("\nFinal Lists for RoBERTa:")
print(f"Cosine Similarity Before Dropout: {cosine_sim_before_dropout}")
print(f"Cosine Similarity First Sample After Dropout: {cosine_sim_first_sample}")
print(f"Average Cosine Similarity of 50 Samples: {avg_cosine_sim_list}")
print(f"Variance of Cosine Similarity of 50 Samples: {var_cosine_sim_list}")
print(f"Average F1 Score of 50 Samples: {avg_f1_score_list}")
print(f"Variance of F1 Score of 50 Samples: {var_f1_score_list}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Streaming output truncated to the last 5000 lines.
Model: RoBERTa
Predicted Answer: 
Maximum Cosine Similarity Before MC Dropout: 1.0000

Predicted Answers from MC Dropout Samples:
Sample 1:  Abercrombie (Cosine Similarity: 0.0000, F1 Score: 0.0000)
Sample 2:  (Cosine Similarity: 1.0000, F1 Score: 1.0000)
Sample 3:  (Cosine Similarity: 1.0000, F1 Score: 1.0000)
Sample 4:  (Cosine Similarity: 1.0000, F1 Score: 1.0000)
Sample 5:  (Cosine Similarity: 1.0000, F1 Score: 1.0000)
Sample 6:  Abercrombie (Cosine Similarity: 0.0000, F1 Score: 0.0000)
Sample 7:  (Cosine Similarity: 1.0000, F1 Score: 1.0000)
Sample 8: Who refused to act until Washington approved plans?The new British command was not in place until July. When he arrived in Albany, Abercrombie (Cosine Similarity: 0.0000, F1 Score: 0.0000)
Sample 9:  Abercrombie (Cosine Similarity: 0.0000, F1 Score: 0.0000)
Sample 10:  Abercrombie (Cosine Similarity: 0.0000, F1 Score: 0.0000)

Question 5709:
Context: The new British command was not i